In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, LongType
)
import uuid

CATALOG = "ecommerce_lakehouse"
MONITOR_TABLE = f"{CATALOG}.gold.pipeline_monitoring"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

run_id = str(uuid.uuid4())
recorded_at = datetime.utcnow()

schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("recorded_at", TimestampType(), False),
    StructField("layer", StringType(), False),
    StructField("entity", StringType(), False),
    StructField("row_count", LongType(), True),
    StructField("null_event_ids", LongType(), True),
    StructField("duplicate_event_ids", LongType(), True),
    StructField("quarantine_count", LongType(), True),
    StructField("latest_event_timestamp", TimestampType(), True),
    StructField("status", StringType(), False),
    StructField("details", StringType(), True)
])

records = []
failures = []

def exists(table):
    return spark.catalog.tableExists(table)

def record(
    layer,
    entity,
    row_count=None,
    null_ids=None,
    duplicate_ids=None,
    quarantine_count=None,
    latest_timestamp=None,
    status="PASS",
    details=""
):
    records.append((
        run_id,
        recorded_at,
        layer,
        entity,
        row_count,
        null_ids,
        duplicate_ids,
        quarantine_count,
        latest_timestamp,
        status,
        details
    ))

# Batch tables
batch_tables = [
    ("bronze", "customers"),
    ("bronze", "products"),
    ("bronze", "orders"),
    ("bronze", "payments"),
    ("silver", "customers"),
    ("silver", "products"),
    ("silver", "orders"),
    ("silver", "payments"),
    ("gold", "daily_sales"),
    ("gold", "product_performance"),
    ("gold", "customer_value"),
    ("gold", "dim_customer_scd2")
]

for layer, name in batch_tables:

    table = f"{CATALOG}.{layer}.{name}"

    if exists(table):
        count = spark.table(table).count()
        record(layer, name, row_count=count)
    else:
        record(
            layer, name,
            status="FAIL",
            details=f"Missing table: {table}"
        )
        failures.append(table)

# Streaming tables
topics = ["orders", "payments", "clickstream"]

for topic in topics:

    bronze_table = (
        f"{CATALOG}.bronze.streaming_{topic}"
    )

    silver_table = (
        f"{CATALOG}.silver.streaming_{topic}"
    )

    quarantine_table = (
        f"{CATALOG}.silver.streaming_quarantine_{topic}"
    )

    if not exists(bronze_table):
        record(
            "bronze_streaming", topic,
            status="FAIL",
            details="Bronze table missing"
        )
        failures.append(bronze_table)
        continue

    bronze_count = spark.table(bronze_table).count()

    record(
        "bronze_streaming",
        topic,
        row_count=bronze_count
    )

    if not exists(silver_table):
        record(
            "silver_streaming", topic,
            status="FAIL",
            details="Silver table missing"
        )
        failures.append(silver_table)
        continue

    silver = spark.table(silver_table)

    silver_count = silver.count()

    null_ids = silver.filter(
        F.col("event_id").isNull()
    ).count()

    duplicate_ids = (
        silver.groupBy("event_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    latest_timestamp = silver.agg(
        F.max("event_timestamp").alias("latest")
    ).first()["latest"]

    quarantine_count = (
        spark.table(quarantine_table).count()
        if exists(quarantine_table)
        else None
    )

    status = "PASS"
    details = "Streaming quality checks passed"

    if null_ids > 0 or duplicate_ids > 0:
        status = "FAIL"
        details = "Null or duplicate event IDs detected"
        failures.append(silver_table)

    record(
        "silver_streaming",
        topic,
        row_count=silver_count,
        null_ids=null_ids,
        duplicate_ids=duplicate_ids,
        quarantine_count=quarantine_count,
        latest_timestamp=latest_timestamp,
        status=status,
        details=details
    )

# Streaming Gold tables
gold_tables = [
    "streaming_daily_sales",
    "streaming_daily_payments",
    "streaming_daily_clickstream"
]

for name in gold_tables:

    table = f"{CATALOG}.gold.{name}"

    if exists(table):
        record(
            "gold_streaming",
            name,
            row_count=spark.table(table).count()
        )
    else:
        record(
            "gold_streaming",
            name,
            status="FAIL",
            details=f"Missing table: {table}"
        )
        failures.append(table)

# Persist metrics
metrics_df = spark.createDataFrame(
    records,
    schema=schema
)

(
    metrics_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(MONITOR_TABLE)
)

print("Monitoring run:", run_id)

display(
    spark.table(MONITOR_TABLE)
    .filter(F.col("run_id") == run_id)
    .orderBy("layer", "entity")
)

if failures:
    raise Exception(
        "PIPELINE MONITORING FAILED: "
        + ", ".join(failures)
    )

print("PIPELINE MONITORING PASSED")

/home/spark-1ca992d5-6bad-4cdb-bd0e-a7/.ipykernel/64/command-8755871134337416-4197525747:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  recorded_at = datetime.utcnow()


Monitoring run: 670d234e-97f0-4ebc-a448-2b70e1ff7f67


run_id,recorded_at,layer,entity,row_count,null_event_ids,duplicate_event_ids,quarantine_count,latest_event_timestamp,status,details
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,bronze,customers,15,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,bronze,orders,15,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,bronze,payments,15,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,bronze,products,15,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,bronze_streaming,clickstream,10,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,bronze_streaming,orders,10,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,bronze_streaming,payments,10,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,gold,customer_value,3,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,gold,daily_sales,2,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,gold,dim_customer_scd2,9,null,null,null,null,PASS,


PIPELINE MONITORING PASSED


In [0]:
%sql
SELECT *
FROM ecommerce_lakehouse.gold.pipeline_monitoring
WHERE run_id = (
    SELECT run_id
    FROM ecommerce_lakehouse.gold.pipeline_monitoring
    ORDER BY recorded_at DESC
    LIMIT 1
)
ORDER BY layer, entity;

run_id,recorded_at,layer,entity,row_count,null_event_ids,duplicate_event_ids,quarantine_count,latest_event_timestamp,status,details
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,bronze,customers,15,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,bronze,orders,15,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,bronze,payments,15,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,bronze,products,15,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,bronze_streaming,clickstream,10,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,bronze_streaming,orders,10,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,bronze_streaming,payments,10,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,gold,customer_value,3,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,gold,daily_sales,2,null,null,null,null,PASS,
670d234e-97f0-4ebc-a448-2b70e1ff7f67,2026-10-31T09:22:42.644Z,gold,dim_customer_scd2,9,null,null,null,null,PASS,


In [0]:
%sql
SELECT *
FROM ecommerce_lakehouse.gold.pipeline_monitoring
WHERE status = 'FAIL'
ORDER BY recorded_at DESC;

run_id,recorded_at,layer,entity,row_count,null_event_ids,duplicate_event_ids,quarantine_count,latest_event_timestamp,status,details


In [0]:

%sql
SELECT
    recorded_at,
    entity,
    row_count,
    null_event_ids,
    duplicate_event_ids,
    quarantine_count,
    latest_event_timestamp,
    status
FROM ecommerce_lakehouse.gold.pipeline_monitoring
WHERE layer = 'silver_streaming'
ORDER BY recorded_at DESC;

recorded_at,entity,row_count,null_event_ids,duplicate_event_ids,quarantine_count,latest_event_timestamp,status
2026-10-31T09:22:42.644Z,orders,5,0,0,0,2026-10-31T08:25:10.128Z,PASS
2026-10-31T09:22:42.644Z,payments,5,0,0,0,2026-10-31T08:25:10.260Z,PASS
2026-10-31T09:22:42.644Z,clickstream,5,0,0,0,2026-10-31T08:25:10.370Z,PASS


In [0]:
%sql
SELECT
    recorded_at,
    layer,
    entity,
    row_count
FROM ecommerce_lakehouse.gold.pipeline_monitoring
ORDER BY recorded_at DESC;

recorded_at,layer,entity,row_count
2026-10-31T09:22:42.644Z,gold_streaming,streaming_daily_payments,0
2026-10-31T09:22:42.644Z,silver_streaming,orders,5
2026-10-31T09:22:42.644Z,bronze_streaming,clickstream,10
2026-10-31T09:22:42.644Z,gold_streaming,streaming_daily_sales,0
2026-10-31T09:22:42.644Z,gold,customer_value,3
2026-10-31T09:22:42.644Z,bronze_streaming,orders,10
2026-10-31T09:22:42.644Z,gold,product_performance,3
2026-10-31T09:22:42.644Z,silver_streaming,payments,5
2026-10-31T09:22:42.644Z,silver_streaming,clickstream,5
2026-10-31T09:22:42.644Z,silver,payments,5


In [0]:
%sql
SELECT _kafka_topic, COUNT(*) AS total_rows,
       MAX(_kafka_offset) AS latest_offset,
       COUNT_IF(_kafka_offset >= 10) AS new_test_rows
FROM ecommerce_lakehouse.bronze.streaming_clickstream
GROUP BY _kafka_topic;

_kafka_topic,total_rows,latest_offset,new_test_rows
clickstream,10,9,0
